# LLM Knowledge Distillation - Fine-tuning Notebook

This notebook demonstrates end-to-end knowledge distillation for large language models.

## Overview

Knowledge distillation transfers knowledge from a large "teacher" model to a smaller "student" model, allowing the student to achieve comparable performance with significantly reduced computational requirements.

### Key Components

1. **Teacher Model**: A large, capable model that provides "soft targets"
2. **Student Model**: A smaller model that learns from both ground truth and teacher outputs
3. **Distillation Loss**: Combines cross-entropy with KL divergence
4. **LoRA**: Efficient fine-tuning with low-rank adaptation

### Loss Function

```
loss = α × CE(student_logits, labels) + β × T² × KL(student_soft, teacher_soft)
```

Where:
- `α` (alpha): Weight for cross-entropy loss (typically 0.2-0.4)
- `β` (beta): Weight for KL divergence loss (typically 0.6-0.8)
- `T` (temperature): Softens probability distributions (typically 2.0-5.0)


## 1. Environment Setup

Install required packages for the project.

In [ ]:
# Install required packages
!pip install -q torch transformers datasets accelerate peft bitsandbytes optuna scikit-learn evaluate
!pip install -q matplotlib seaborn plotly streamlit pyyaml tqdm

## 2. Hardware Check

Verify GPU availability and compute capability.

In [ ]:
import torch

# Check GPU availability
print("=" * 60)
print("Hardware Information")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Total memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"  Compute capability: {props.major}.{props.minor}")
        print(f"  BF16 support: {props.major >= 8}")

# Determine optimal dtype
CAN_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if CAN_BF16 else torch.float16
print(f"\nUsing dtype: {DTYPE}")

Hardware Information
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1

GPU 0: Tesla T4
  Total memory: 14.6 GB
  Compute capability: 7.5
  BF16 support: False

Using dtype: torch.float16


## 3. Configuration

Set up model and training configurations.

In [ ]:
# Model Configuration
#TEACHER_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # 7B parameter teacher
#STUDENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B parameter student
TEACHER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B (fits T4 16GB)
STUDENT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # 0.5B (fits anywhere)

# Dataset Configuration
DATASET_NAME = "databricks/databricks-dolly-15k"

# Training Configuration
BLOCK_SIZE = 512
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

# Distillation Configuration
TEMPERATURE = 2.0
ALPHA = 0.3  # CE loss weight
BETA = 0.7   # KD loss weight

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Output Configuration
OUTPUT_DIR = "./artifacts/checkpoints"

print("Configuration:")
print(f"  Teacher: {TEACHER_MODEL}")
print(f"  Student: {STUDENT_MODEL}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Alpha: {ALPHA}, Beta: {BETA}")

Configuration:
  Teacher: Qwen/Qwen2.5-1.5B-Instruct
  Student: Qwen/Qwen2.5-0.5B-Instruct
  Dataset: databricks/databricks-dolly-15k
  Temperature: 2.0
  Alpha: 0.3, Beta: 0.7


## 4. Imports

In [ ]:
import os
import math
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# Set random seed for reproducibility
torch.manual_seed(42)

print("✅ Imports successful")

✅ Imports successful


## 5. Load Dataset

In [ ]:
# Load dataset
print(f"Loading dataset: {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME, split="train")

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train examples: {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")

# Show example
print("\nExample entry:")
example = train_dataset[0]
for key, value in example.items():
    print(f"  {key}: {value[:100] if isinstance(value, str) else value}...")

Loading dataset: databricks/databricks-dolly-15k...
Train examples: 13,509
Validation examples: 1,502

Example entry:
  instruction: Given the reference text below the eruption of Mount Vesuvius, how high was the mountain after the d...
  context: In December 1631, Mount Vesuvius in Italy erupted. The eruption began on 16 December 1631 and culmin...
  response: Mount Vesuvius had a reduced summit by 450 meters....
  category: closed_qa...


## 6. Preprocess Dataset

Format examples for instruction-following training.

In [ ]:
def format_instruction(example):
    """Format an instruction example."""
    instruction = example.get("instruction", "")
    context = example.get("context", "")
    response = example.get("response", "")

    if context:
        prompt = f"### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {
        "text": prompt + response,
        "prompt": prompt,
        "response": response,
    }

# Apply formatting
train_dataset = train_dataset.map(format_instruction)
val_dataset = val_dataset.map(format_instruction)

print("Formatted example:")
print(train_dataset[0]["text"][:500])

Formatted example:
### Instruction:
Given the reference text below the eruption of Mount Vesuvius, how high was the mountain after the disaster?

### Context:
In December 1631, Mount Vesuvius in Italy erupted. The eruption began on 16 December 1631 and culminated the day after. The Volcanic Explosivity Index was VEI-5, and it was a Plinian eruption that buried many villages under the resulting lava flows. It is estimated that between 4,000 people were killed by the eruption, making it the highest death toll for a 


## 7. Load Teacher Model

Load the teacher model with 4-bit quantization to save memory.

In [ ]:
print(f"Loading teacher model: {TEACHER_MODEL}...")

# Quantization config for teacher
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
)

# Load teacher tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
teacher_tokenizer.padding_side = "left"
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

# Load teacher model
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Freeze teacher
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

print(f"✅ Teacher loaded. Parameters: {sum(p.numel() for p in teacher.parameters()) / 1e6:.0f}M")

Loading teacher model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Teacher loaded. Parameters: 889M


## 8. Load Student Model with LoRA

In [ ]:
print(f"Loading student model: {STUDENT_MODEL}...")

# Load student tokenizer
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
student_tokenizer.padding_side = "left"
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

# Load student model
student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    #torch_dtype=DTYPE,
    dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
student.config.use_cache = False  # Disable for training

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

student = get_peft_model(student, lora_config)
student.print_trainable_parameters()

Loading student model: Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 9. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize examples."""
    result = student_tokenizer(
        examples["text"],
        max_length=BLOCK_SIZE,
        padding="max_length",
        truncation=True,
    )
    result["labels"] = result["input_ids"].copy()
    return result

# Tokenize datasets
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print(f"Tokenized train: {len(tokenized_train):,} examples")
print(f"Tokenized val: {len(tokenized_val):,} examples")

Tokenizing datasets...


Map:   0%|          | 0/13509 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenized train: 13,509 examples
Tokenized val: 1,502 examples


## 10. Define Distillation Loss

In [ ]:
def compute_kd_loss(student_logits, teacher_logits, labels, temperature, alpha, beta):
    """
    Compute knowledge distillation loss.

    loss = alpha * CE + beta * T² * KL
    """
    # Cross-entropy loss
    ce_loss = F.cross_entropy(
        student_logits.view(-1, student_logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )

    # KL divergence loss
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)

    with torch.no_grad():
        teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)

    kl_loss = F.kl_div(
        student_log_probs.view(-1, student_logits.size(-1)),
        teacher_probs.view(-1, teacher_logits.size(-1)),
        reduction="batchmean",
        log_target=False,
    )

    kl_loss = kl_loss * (temperature ** 2)

    # Combined loss
    total_loss = alpha * ce_loss + beta * kl_loss

    return total_loss, ce_loss, kl_loss

print("✅ Distillation loss function defined")

✅ Distillation loss function defined


## 11. Custom Trainer for Knowledge Distillation

In [ ]:
class KDTrainer(Trainer):
    """Custom trainer for knowledge distillation."""

    def __init__(self, teacher_model, temperature, alpha, beta, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.beta = beta

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Compute distillation loss."""
        labels = inputs["labels"]

        # Student forward pass
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        student_logits = outputs.logits

        # Teacher forward pass (no gradients)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=inputs["input_ids"].to(self.teacher_model.device),
                attention_mask=inputs["attention_mask"].to(self.teacher_model.device),
            )
            teacher_logits = teacher_outputs.logits.to(student_logits.device)

        # Compute loss
        loss, ce_loss, kl_loss = compute_kd_loss(
            student_logits,
            teacher_logits,
            labels,
            self.temperature,
            self.alpha,
            self.beta,
        )

        # Log individual losses
        if self.state.global_step % 10 == 0:
            self.log({"ce_loss": ce_loss.item(), "kl_loss": kl_loss.item()})

        return (loss, outputs) if return_outputs else loss

print("✅ KDTrainer defined")

✅ KDTrainer defined


## 12. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=CAN_BF16,
    fp16=not CAN_BF16 and torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

print("Training arguments:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Alpha: {ALPHA}, Beta: {BETA}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments:
  Epochs: 1
  Batch size: 1
  Gradient accumulation: 1
  Effective batch size: 1
  Learning rate: 0.0002
  Temperature: 2.0
  Alpha: 0.3, Beta: 0.7


## 13. Create Trainer and Train

In [ ]:
# Create data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=student_tokenizer,
    mlm=False,
)

# Create trainer
trainer = KDTrainer(
    teacher_model=teacher,
    temperature=TEMPERATURE,
    alpha=ALPHA,
    beta=BETA,
    model=student,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("✅ Trainer created. Starting training...")
print("=" * 60)

✅ Trainer created. Starting training...


In [ ]:
# Train
trainer.train()

Step,Training Loss,Validation Loss
1,No log,8.263070
2,8.263071,8.263070
3,8.263071,8.263070
4,8.263071,8.263070
5,8.263071,8.224256
6,8.239165,8.165069
7,8.239165,8.099120
8,8.132675,8.027901
9,8.132675,7.942477
10,7.990287,7.848279


KeyboardInterrupt: 

## 14. Evaluate Model

In [ ]:
# Evaluate on validation set
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
print("=" * 40)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Compute perplexity
if "eval_loss" in eval_results:
    try:
        perplexity = math.exp(eval_results["eval_loss"])
        print(f"  perplexity: {perplexity:.2f}")
    except OverflowError:
        print(f"  perplexity: inf")

## 15. Save Model

In [ ]:
# Save the trained model
SAVE_DIR = "./artifacts/best_model/final"

os.makedirs(SAVE_DIR, exist_ok=True)

# Save student model
student.save_pretrained(SAVE_DIR)
student_tokenizer.save_pretrained(SAVE_DIR)

# Save training configuration
import json
config_dict = {
    "teacher_model": TEACHER_MODEL,
    "student_model": STUDENT_MODEL,
    "temperature": TEMPERATURE,
    "alpha": ALPHA,
    "beta": BETA,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
}

with open(os.path.join(SAVE_DIR, "training_config.json"), "w") as f:
    json.dump(config_dict, f, indent=2)

print(f"✅ Model saved to {SAVE_DIR}")

## 16. Test Inference

In [ ]:
# Test generation
test_prompts = [
    "Explain the concept of machine learning in simple terms.",
    "What is knowledge distillation in AI?",
    "Write a Python function to calculate factorial.",
]

student.eval()

print("\nTest Generation:")
print("=" * 60)

for prompt in test_prompts:
    # Format prompt
    formatted_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"

    # Tokenize
    inputs = student_tokenizer(formatted_prompt, return_tensors="pt").to(student.device)

    # Generate
    with torch.no_grad():
        outputs = student.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=student_tokenizer.pad_token_id,
            eos_token_id=student_tokenizer.eos_token_id,
        )

    # Decode
    generated_text = student_tokenizer.decode(
        outputs[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\nPrompt: {prompt}")
    print(f"Response: {generated_text}")
    print("-" * 60)

## 17. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Extract log history
log_history = trainer.state.log_history

# Extract metrics
train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
if train_loss:
    axes[0].plot(train_steps, train_loss, label="Training Loss", color="blue")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Validation loss
if eval_loss:
    axes[1].plot(eval_steps, eval_loss, label="Validation Loss", color="orange")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Validation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./artifacts/plots/training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Training curves saved to ./artifacts/plots/training_curves.png")

## 18. Summary

This notebook demonstrated:

1. ✅ Loading and preprocessing instruction-following datasets
2. ✅ Loading teacher model with quantization
3. ✅ Loading student model with LoRA
4. ✅ Implementing knowledge distillation loss (CE + KL)
5. ✅ Training with custom KDTrainer
6. ✅ Evaluating and saving the model
7. ✅ Testing inference

### Next Steps

1. **Hyperparameter Optimization**: Run Optuna to find optimal temperature, alpha, beta, and LoRA parameters
2. **Extended Training**: Train for more epochs with larger datasets
3. **Evaluation**: Compare distilled student against baseline (no distillation)
4. **Deployment**: Export model for production use

### Key Learnings

- Knowledge distillation transfers reasoning patterns from teacher to student
- Temperature scaling reveals relationships between tokens
- LoRA enables efficient fine-tuning with minimal parameters
- The combined loss (CE + KD) often outperforms either alone